
# C 방식: 제품 × 여러명 배치 프롬프트 생성 (v4)
입력
- `persona_attributes_weighted.jsonl` (가중치+meta 포함)
- `product_info_preprocessed.jsonl` (prompt_block 포함)

출력
- `prompts_C.jsonl`
- `prompts_C_preview.json` (샘플 확인용)


In [1]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("persona_attributes_weighted.jsonl")
PRODUCT_JSONL = Path("product_info_preprocessed.jsonl")

OUT_JSONL     = Path("prompts_C.jsonl")
OUT_PREVIEW   = Path("prompts_C_preview.json")

BATCH_SIZE = 10   # 한 프롬프트에 들어갈 페르소나 수 (5~30 권장)
ATTR_LIMIT = 40   # 페르소나 속성 표시 최대 수
print("CONFIG loaded.")

CONFIG loaded.


In [2]:

# =============================
# 1) Load data
# =============================
import json

# Personas
personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            personas.append(json.loads(line))

# Products
products = []
with open(PRODUCT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            products.append(json.loads(line))

print("Loaded:", len(personas), "personas /", len(products), "products")

Loaded: 363 personas / 15 products


In [ ]:
def persona_to_prompt_block(p: Dict[str, Any]) -> str:
    meta = p.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", meta.get("Description",""))

    # 속성 문자열을 미리 만들어서 f-string에 변수로 주입
    attrs_str = format_attributes_for_prompt(p.get("attributes", {}), limit=ATTR_LIMIT)

    # 클러스터 블록도 미리 완성
    if (cluster or label or desc):
        cluster_block = f"- cluster: {cluster}\n- label: {label}\n- desc: {desc}"
    else:
        cluster_block = "- cluster: N/A"

    return (
        "[페르소나]\n"
        f"- id: {p.get('persona_key','')}\n"
        "- 속성(가중치 합=1):\n"
        f"{attrs_str}\n"
        "- 클러스터 컨텍스트:\n"
        f"{cluster_block}"
    ).strip()


def build_batch_prompt(product: Dict[str, Any], persona_batch: List[Dict[str, Any]]) -> str:
    product_block = product.get("prompt_block") or ""

    # ⚠️ 여기서 '\n\n'.join 을 f-string 밖에서 먼저 수행
    persona_blocks = [persona_to_prompt_block(p) for p in persona_batch]
    persona_blocks_str = "\n\n".join(persona_blocks)

    header = (
        "[역할]\n"
        "당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.\n"
        "아래의 \"제품 정보\"와 \"페르소나 목록\"을 바탕으로, 각 페르소나마다\n"
        "해당 제품의 구매자 페르소나를 **싱글 턴**으로 완결된 JSON 객체로 생성하세요.\n"
        "각 페르소나는 서로 독립적이며, 서로의 정보에 영향을 주지 마세요.\n\n"
        "[제품 정보]\n"
        f"{product_block}\n\n"
        "[페르소나 목록]\n"
        f"{persona_blocks_str}\n\n"
        "[규칙]\n"
        "- '클러스터 컨텍스트'는 페르소나의 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.\n"
        "- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.\n"
        "- 추석/설, 광고/프로모션/계절성을 반영합니다.\n"
        "- **반드시 아래 JSON 스키마(JSON 배열)를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.\n\n"
        "[출력 스키마(JSON 배열)]\n"
        "[\n"
        "  {\n"
        '    "persona_id": "p_{{product_id_or_name}}_{{persona_key}}",\n'
        f'    "product_name": "{product.get("product_name","")}",\n'
        f'    "product_id": "{product.get("product_id","")}",\n'
        '    "segment_ref": "{{persona_key}}",\n'
        '    "attributes": { "{{속성명}}": {"value": "<값>", "weight": <0~1> }, "...": "..." },\n'
        '    "purchase_pattern": {\n'
        '      "avg_purchase_prob": <0~1>,\n'
        '      "avg_purchase_qty": <int>,\n'
        '      "seasonality": {"추석": "+x%", "설": "+y%"},\n'
        '      "promotion_effect": "광고/프로모션 노출 시 +z%"\n'
        "    },\n"
        '    "forecast_12mo": {\n'
        '      "2024-07": {"prob": <0~1>, "qty": <int>},\n'
        '      "...": {},\n'
        '      "2025-06": {"prob": <0~1>, "qty": <int>}\n'
        "    }\n"
        "  },\n"
        "  ...\n"
        "]"
    )
    return header.strip()


SyntaxError: f-string expression part cannot include a backslash (2892690554.py, line 77)

In [ ]:

# =============================
# 3) Build & save
# =============================
import json
from pathlib import Path

records = []
for prod in products:
    pid_or_name = prod.get("product_id") or (prod.get("product_name","") or "").replace(" ", "_")
    for batch in chunked(personas, BATCH_SIZE):
        prompt_text = build_batch_prompt({**prod, "product_id_or_name": pid_or_name}, batch)
        records.append({
            "product": {"product_id_or_name": pid_or_name, "product_name": prod.get("product_name")},
            "personas": [{"persona_key": p.get("persona_key")} for p in batch],
            "prompt": prompt_text
        })

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Path(OUT_PREVIEW).write_text(json.dumps(records[:1], ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUT_JSONL, "size=", Path(OUT_JSONL).stat().st_size, "bytes")
len(records)